# 01 — Bronze: clients

Mesmo contrato do `00_bronze_bids`: ingerir como está, validar o conjunto de
colunas, deixar todo julgamento para o Silver.

A exportação de clientes carrega duas peculiaridades que vale notar mas
*não* corrigir aqui — um sentinela `2999-12-31` marcando contratos sem prazo
definido, e strings literais `'null'` onde falta um valor. As duas
sobrevivem no Bronze intocadas.

Mesma abordagem sem bibliotecas de cluster do `00_bronze_bids`: pandas +
openpyxl lê o arquivo, depois é convertido para um Spark DataFrame.

In [0]:
%pip install openpyxl

In [0]:
CATALOG = "bronze"
SCHEMA = "bid"
VOLUME_PATH = "/Volumes/raw/bid/bids/clients.xlsx"
TABLE = f"{CATALOG}.{SCHEMA}.clients"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
import pandas as pd
from pyspark.sql import functions as F

pdf_raw = pd.read_excel(VOLUME_PATH, sheet_name="Clients", dtype=str)
df_raw = spark.createDataFrame(pdf_raw)

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(VOLUME_PATH))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(TABLE)
)

print(f"linhas: {spark.table(TABLE).count()}")

In [0]:
EXPECTED = {
    "client_id", "contract_name", "status", "start_date", "end_date",
    "state", "city", "segment", "account_executive", "director",
    "manager", "coordinator",
}

actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}
missing, unexpected = EXPECTED - actual, actual - EXPECTED

if missing:
    raise ValueError(f"Colunas ausentes na fonte: {sorted(missing)}")
if unexpected:
    print(f"AVISO — novas colunas absorvidas, revisar Silver: {sorted(unexpected)}")

print("Checagem de schema aprovada.")